# 06 - Recommendations Data Wrangling

Notebook ini memproses dataset `recommendations.csv`.

Tabel ini menyimpan rekomendasi yang dapat berasal dari konteks harian atau mingguan. Oleh karena itu, validasi utama diarahkan pada `period_type` dan pasangan source id yang digunakan.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Mengatur tampilan dataframe agar output notebook lebih mudah dibaca.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Cari root project otomatis
# Mengambil lokasi kerja notebook saat ini.
current_path = Path.cwd().resolve()

# Menelusuri parent folder sampai menemukan root project yang memiliki folder data/raw.
for path in [current_path] + list(current_path.parents):
    if (path / "data" / "raw").exists():
        PROJECT_ROOT = path
        break

# Menentukan folder sumber data raw.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
# Menentukan folder output data hasil cleaning.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# Menentukan folder output report dan validation summary.
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

# Membuat folder processed jika belum tersedia.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# Membuat folder reports jika belum tersedia.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Menampilkan path project untuk memastikan notebook membaca folder yang benar.
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORT_DIR   :", REPORT_DIR)

PROJECT_ROOT : C:\Data Codingan\student_stress_data_science
RAW_DIR      : C:\Data Codingan\student_stress_data_science\data\raw
PROCESSED_DIR: C:\Data Codingan\student_stress_data_science\data\processed
REPORT_DIR   : C:\Data Codingan\student_stress_data_science\outputs\reports


## 1. Load Dataset

In [ ]:
# memuat dataset dari folder yang sesuai dan menampilkan sampel awal data.
# Membaca file CSV ke dalam dataframe.
recommendations = pd.read_csv(RAW_DIR / "recommendations.csv")
users_clean = pd.read_csv(PROCESSED_DIR / "users_clean.csv")
stress_predictions_clean = pd.read_csv(PROCESSED_DIR / "stress_predictions_clean.csv")
weekly_summaries_clean = pd.read_csv(PROCESSED_DIR / "weekly_summaries_clean.csv")

# Menampilkan beberapa baris awal untuk memahami bentuk data.
recommendations.head()

,id,user_id,stress_prediction_id,weekly_summary_id,period_type,category,title,recommendation_text,priority_level,created_at
0,1,1,1.0,NaN,daily,mood_regulation,Stabilkan mood,Mood lo sedang rendah. Coba journaling singkat...,Medium,2026-01-02 00:03:00
1,2,1,2.0,NaN,daily,workload,Atur prioritas tugas,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,High,2026-01-02 20:48:00
2,3,1,3.0,NaN,daily,digital_habit,Batasi screen time,Screen time lo tinggi. Kurangi penggunaan laya...,Medium,2026-01-03 20:54:00
3,4,1,4.0,NaN,daily,mood_regulation,Stabilkan mood,Mood lo sedang rendah. Coba journaling singkat...,Medium,2026-01-05 01:01:00
4,5,1,5.0,NaN,daily,workload,Atur prioritas tugas,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,High,2026-01-05 23:14:00


## 2. Assessing Data

In [ ]:
# Menampilkan struktur kolom, tipe data, dan jumlah non-null.
recommendations.info()

<class 'pandas.DataFrame'>
RangeIndex: 27198 entries, 0 to 27197
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    27198 non-null  int64  
 1   user_id               27198 non-null  int64  
 2   stress_prediction_id  26365 non-null  float64
 3   weekly_summary_id     833 non-null    float64
 4   period_type           27198 non-null  str    
 5   category              27198 non-null  str    
 6   title                 27198 non-null  str    
 7   recommendation_text   27198 non-null  str    
 8   priority_level        27198 non-null  str    
 9   created_at            27198 non-null  str    
dtypes: float64(2), int64(2), str(6)
memory usage: 2.1 MB


In [ ]:
# Menampilkan ringkasan statistik untuk kolom numerik dan kategorikal.
recommendations.describe(include='all')

,id,user_id,stress_prediction_id,weekly_summary_id,period_type,category,title,recommendation_text,priority_level,created_at
count,27198.000000,27198.000000,26365.000000,833.000000,27198,27198,27198,27198,27198,27198
unique,NaN,NaN,NaN,NaN,4,11,11,11,3,18669
top,NaN,NaN,NaN,NaN,daily,workload,Atur prioritas tugas,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,High,2026-03-18 21:05:00
freq,NaN,NaN,NaN,NaN,26287,12197,12197,12197,15086,158
mean,13599.500000,150.505147,13505.331766,1782.460984,NaN,NaN,NaN,NaN,NaN,NaN
std,7851.530647,86.662853,7796.995406,1051.702572,NaN,NaN,NaN,NaN,NaN,NaN
min,1.000000,1.000000,1.000000,10.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,6800.250000,75.000000,6754.000000,827.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,13599.500000,151.000000,13528.000000,1834.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,20398.750000,226.000000,20258.000000,2685.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# menilai missing value, duplicate, dan kandidat masalah kualitas data.
print("Missing value:")
# Menghitung jumlah missing value pada setiap kolom.
print(recommendations.isna().sum())

# Membersihkan whitespace dan menstandarkan format teks.
period_type = recommendations["period_type"].astype(str).str.strip().str.lower()

daily_rows = period_type == "daily"
weekly_rows = period_type == "weekly"

daily_source_valid = (
    recommendations.loc[daily_rows, "stress_prediction_id"].notna()
    & recommendations.loc[daily_rows, "weekly_summary_id"].isna()
)

weekly_source_valid = (
    recommendations.loc[weekly_rows, "weekly_summary_id"].notna()
    & recommendations.loc[weekly_rows, "stress_prediction_id"].isna()
)

print("\nPeriod type unique:")
# Melihat variasi nilai unik untuk menilai konsistensi kategori atau format.
print(recommendations["period_type"].unique())

print("\nDaily source valid:", daily_source_valid.all())
print("Weekly source valid:", weekly_source_valid.all())

Missing value:
id                          0
user_id                     0
stress_prediction_id      833
weekly_summary_id       26365
period_type                 0
category                    0
title                       0
recommendation_text         0
priority_level              0
created_at                  0
dtype: int64

Period type unique:
<StringArray>
['daily', 'Daily', 'weekly', 'Weekly']
Length: 4, dtype: str

Daily source valid: True
Weekly source valid: True


## Insight:

Missing value pada `stress_prediction_id` atau `weekly_summary_id` harus dinilai berdasarkan `period_type`, bukan hanya berdasarkan jumlah missing secara umum. Untuk baris daily, `weekly_summary_id` memang tidak digunakan. Untuk baris weekly, `stress_prediction_id` memang tidak digunakan.

Tindakan cleaning yang dilakukan adalah standardisasi teks, validasi `period_type`, validasi `user_id`, serta memastikan rekomendasi hanya merujuk ke source id yang tersedia pada tabel processed terkait.

## 3. Cleaning Data

Langkah cleaning:

1. Mengubah key dan source id ke tipe numerik.
2. Menstandarkan `period_type` ke lowercase.
3. Membersihkan teks pada kategori, judul, dan isi rekomendasi.
4. Menghapus baris dengan informasi umum yang tidak valid.
5. Memastikan `user_id` valid.
6. Memvalidasi aturan source id berdasarkan `period_type`.

In [ ]:
# membuat salinan dataframe lalu menjalankan proses cleaning sesuai hasil assessing.
recommendations_clean = recommendations.copy()

# Mengubah kolom ke tipe numerik; nilai yang gagal dikonversi menjadi NaN.
recommendations_clean["id"] = pd.to_numeric(recommendations_clean["id"], errors="coerce")
recommendations_clean["user_id"] = pd.to_numeric(recommendations_clean["user_id"], errors="coerce")
recommendations_clean["stress_prediction_id"] = pd.to_numeric(recommendations_clean["stress_prediction_id"], errors="coerce")
recommendations_clean["weekly_summary_id"] = pd.to_numeric(recommendations_clean["weekly_summary_id"], errors="coerce")
# Membersihkan whitespace dan menstandarkan format teks.
recommendations_clean["period_type"] = recommendations_clean["period_type"].astype(str).str.strip().str.lower()
recommendations_clean["category"] = recommendations_clean["category"].astype(str).str.strip()
recommendations_clean["title"] = recommendations_clean["title"].astype(str).str.strip()
recommendations_clean["recommendation_text"] = recommendations_clean["recommendation_text"].astype(str).str.strip()
recommendations_clean["priority_level"] = recommendations_clean["priority_level"].astype(str).str.strip().str.title()
# Mengubah kolom ke tipe datetime; format yang tidak valid menjadi NaT.
recommendations_clean["created_at"] = pd.to_datetime(recommendations_clean["created_at"], errors="coerce")

# Menghapus baris yang kehilangan kolom kunci atau informasi penting.
recommendations_clean = recommendations_clean.dropna(
    subset=["id", "user_id", "period_type", "category", "title", "recommendation_text", "priority_level", "created_at"]
)

recommendations_clean = recommendations_clean[
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    recommendations_clean["user_id"].isin(set(users_clean["id"]))
]

recommendations_clean = recommendations_clean[
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    recommendations_clean["period_type"].isin(["daily", "weekly"])
]

valid_prediction_ids = set(stress_predictions_clean["id"])
valid_weekly_ids = set(weekly_summaries_clean["id"])

daily_mask = recommendations_clean["period_type"] == "daily"
weekly_mask = recommendations_clean["period_type"] == "weekly"

daily_valid = (
    daily_mask
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    & recommendations_clean["stress_prediction_id"].isin(valid_prediction_ids)
    & recommendations_clean["weekly_summary_id"].isna()
)

weekly_valid = (
    weekly_mask
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    & recommendations_clean["weekly_summary_id"].isin(valid_weekly_ids)
    & recommendations_clean["stress_prediction_id"].isna()
)

recommendations_clean = recommendations_clean[daily_valid | weekly_valid]

recommendations_clean["id"] = recommendations_clean["id"].astype(int)
recommendations_clean["user_id"] = recommendations_clean["user_id"].astype(int)
recommendations_clean["stress_prediction_id"] = recommendations_clean["stress_prediction_id"].astype("Int64")
recommendations_clean["weekly_summary_id"] = recommendations_clean["weekly_summary_id"].astype("Int64")
recommendations_clean["created_at"] = recommendations_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

recommendations_clean = recommendations_clean[
    [
        "id", "user_id", "stress_prediction_id", "weekly_summary_id",
        "period_type", "category", "title", "recommendation_text",
        "priority_level", "created_at"
    ]
# Mengurutkan data agar proses deduplikasi atau output lebih stabil.
].sort_values("id")

# Menampilkan beberapa baris awal untuk memahami bentuk data.
recommendations_clean.head()

,id,user_id,stress_prediction_id,weekly_summary_id,period_type,category,title,recommendation_text,priority_level,created_at
0,1,1,1,<NA>,daily,mood_regulation,Stabilkan mood,Mood lo sedang rendah. Coba journaling singkat...,Medium,2026-01-02 00:03:00
1,2,1,2,<NA>,daily,workload,Atur prioritas tugas,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,High,2026-01-02 20:48:00
2,3,1,3,<NA>,daily,digital_habit,Batasi screen time,Screen time lo tinggi. Kurangi penggunaan laya...,Medium,2026-01-03 20:54:00
3,4,1,4,<NA>,daily,mood_regulation,Stabilkan mood,Mood lo sedang rendah. Coba journaling singkat...,Medium,2026-01-05 01:01:00
4,5,1,5,<NA>,daily,workload,Atur prioritas tugas,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,High,2026-01-05 23:14:00


## Insight Setelah Cleaning:

`recommendations_clean` hanya berisi rekomendasi yang memiliki struktur source valid. Dengan demikian, rekomendasi harian tetap terhubung ke prediction harian, sedangkan rekomendasi mingguan tetap terhubung ke weekly summary.

## 4. Validation dan Save Output

In [ ]:
# membuat tabel validasi untuk memastikan hasil cleaning memenuhi aturan kualitas data.
daily_mask = recommendations_clean["period_type"] == "daily"
weekly_mask = recommendations_clean["period_type"] == "weekly"

daily_valid = (
    recommendations_clean.loc[daily_mask, "stress_prediction_id"].notna()
    & recommendations_clean.loc[daily_mask, "weekly_summary_id"].isna()
).all()

weekly_valid = (
    recommendations_clean.loc[weekly_mask, "weekly_summary_id"].notna()
    & recommendations_clean.loc[weekly_mask, "stress_prediction_id"].isna()
).all()

# Membuat dataframe validasi untuk mendokumentasikan hasil pengecekan kualitas data.
validation = pd.DataFrame([
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    {"rule": "period_type valid", "passed": recommendations_clean["period_type"].isin(["daily", "weekly"]).all()},
    {"rule": "daily source valid", "passed": daily_valid},
    {"rule": "weekly source valid", "passed": weekly_valid},
])

validation

,rule,passed
0,period_type valid,True
1,daily source valid,True
2,weekly source valid,True


In [ ]:
# menyimpan output hasil cleaning atau report ke folder tujuan.
# Menyimpan dataframe ke file CSV.
recommendations_clean.to_csv(PROCESSED_DIR / "recommendations_clean.csv", index=False)
validation.to_csv(REPORT_DIR / "recommendations_validation.csv", index=False)

print("Saved:", PROCESSED_DIR / "recommendations_clean.csv")

Saved: C:\Data Codingan\student_stress_data_science\data\processed\recommendations_clean.csv
